# Avance 2 — Análisis Estadístico Completo
## Variabilidad Hospitalaria en Intensidad de Procedimientos Médicos

**Pregunta de investigación:** ¿Varía significativamente la intensidad de procedimientos médicos entre hospitales públicos para pacientes con diagnóstico de neoplasia o sepsis, y está esa variación asociada a diferencias en mortalidad intrahospitalaria y días de estadía?

---

### Grupos diagnósticos
- **Neoplasias** (atención programada): C50, C18–C20, C53, C34
- **Sepsis** (urgencia): A40, A41

### Hipótesis
- **H1:** Variabilidad inter-hospital significativa en n_procedimientos (Kruskal-Wallis + Dunn-Bonferroni)
- **H2:** Intensidad de procedimientos asociada a mortalidad intrahospitalaria (Regresión logística + efectos fijos)
- **H3:** Intensidad de procedimientos predice días de estadía independientemente de la severidad (OLS + efectos fijos)

---
**Datos:** GRD Público FONASA 2019–2024 (~5.8M registros)

## 0. Configuración e importaciones

In [ ]:
# Librerías estándar y científicas
import os
import sys
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Agregar advance_2/ al path para importar config.py y utils.py
ADVANCE2_DIR = os.path.abspath('..')
if ADVANCE2_DIR not in sys.path:
    sys.path.insert(0, ADVANCE2_DIR)

# Importar configuración centralizada y funciones utilitarias
from config import (
    ALPHA, COL_HOSPITAL, COL_SEVERIDAD, COL_PESO,
    DIR_GRAFICOS, DIR_TABLAS, DIR_MODELOS,
    SEMILLA, SHAPIRO_N, CODIGOS_NEOPLASIA, CODIGOS_SEPSIS, sig_etiqueta,
)
from utils import (
    cargar_datos_grd, derivar_variables, filtrar_grupos_diagnosticos,
    limpiar_datos, tabla_completitud, liberar_memoria,
)

# Crear carpetas de salida si no existen
os.makedirs(DIR_TABLAS, exist_ok=True)
os.makedirs(DIR_GRAFICOS, exist_ok=True)
os.makedirs(DIR_MODELOS, exist_ok=True)

# Estilo de gráficos
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

DPI = 150  # DPI reducido para visualización en notebook

# Columnas que necesitamos del dataset (no leemos todo para ahorrar memoria)
COLUMNAS = [
    COL_HOSPITAL, 'FECHA_NACIMIENTO', 'FECHA_INGRESO', 'FECHAALTA',
    'TIPOALTA', 'DIAGNOSTICO1', 'IR_29301_SEVERIDAD', 'IR_29301_PESO',
] + [f'PROCEDIMIENTO{i}' for i in range(1, 31)]

print('Configuración lista.')
print(f'Códigos neoplasia: {CODIGOS_NEOPLASIA}')
print(f'Códigos sepsis:    {CODIGOS_SEPSIS}')
print(f'Nivel de significancia α = {ALPHA}')

## 1. Carga de Datos

Se cargan todos los archivos GRD Público (2019–2024) con `dtype=str` para
minimizar el uso de memoria en la carga inicial. Luego se derivan las variables analíticas.

In [ ]:
print('Cargando datos GRD (esto puede tomar unos minutos)...')
datos_crudos = cargar_datos_grd(columnas=COLUMNAS)
print(f'Total de filas cargadas: {len(datos_crudos):,}')
print(f'Columnas disponibles: {len(datos_crudos.columns)}')

In [ ]:
# Derivar variables analíticas:
#   edad = año ingreso - año nacimiento (aproximación)
#   dias_estadia = FECHAALTA - FECHA_INGRESO (días)
#   n_procedimientos = cantidad de procedimientos CIE-9 no nulos
#   n_proc_unicos = diversidad de tipos de procedimiento
#   mortalidad = 1 si TIPOALTA == 'FALLECIDO'
datos_crudos = derivar_variables(datos_crudos)

print('Variables derivadas exitosamente. Muestra:')
print(datos_crudos[['edad', 'dias_estadia', 'n_procedimientos', 'n_proc_unicos', 'mortalidad']].describe().round(2))

## 2. Filtrado por Grupo Diagnóstico

Se filtra por `DIAGNOSTICO1` para obtener los dos grupos de análisis.

In [ ]:
df_neo, df_sep = filtrar_grupos_diagnosticos(datos_crudos)
liberar_memoria(datos_crudos)  # Liberar ~5.8M filas de memoria

print(f'Neoplasia (antes de limpieza): {len(df_neo):,} registros')
print(f'Sepsis (antes de limpieza):    {len(df_sep):,} registros')

print('\nTop diagnósticos — Neoplasia:')
print(df_neo['DIAGNOSTICO1'].value_counts().head(6).to_string())
print('\nTop diagnósticos — Sepsis:')
print(df_sep['DIAGNOSTICO1'].value_counts().head(4).to_string())

## 3. Limpieza de Datos

**Decisiones documentadas:**
1. **Sin COD_HOSPITAL** → eliminar (no asignable a ningún hospital)
2. **Sin IR_29301_SEVERIDAD** → eliminar (covariable clave del modelo)
3. **dias_estadia < 0** → eliminar (error de codificación: alta antes del ingreso)
4. **dias_estadia > P99** → eliminar (estadías extremas clínicamente distintas, por grupo)
5. **Severidad no numérica** → eliminar tras coerción
6. **Hospital con < 30 casos** → eliminar (poder estadístico insuficiente)

In [ ]:
n_neo_crudo = len(df_neo)
n_sep_crudo = len(df_sep)

df_neo = limpiar_datos(df_neo, 'neoplasia')
df_sep = limpiar_datos(df_sep, 'sepsis')

print(f'\nNeoplasia: {n_neo_crudo:,} → {len(df_neo):,} ({100*len(df_neo)/n_neo_crudo:.1f}% retenido)')
print(f'Sepsis:    {n_sep_crudo:,} → {len(df_sep):,} ({100*len(df_sep)/n_sep_crudo:.1f}% retenido)')

### 3.1 Tabla de Completitud

In [ ]:
# Calcular completitud para variables clave del análisis
cols_clave = [COL_HOSPITAL, 'edad', 'dias_estadia', 'n_procedimientos',
              'n_proc_unicos', 'mortalidad', COL_SEVERIDAD, COL_PESO]

comp_neo = tabla_completitud(df_neo[[c for c in cols_clave if c in df_neo.columns]], 'neoplasia')
comp_sep = tabla_completitud(df_sep[[c for c in cols_clave if c in df_sep.columns]], 'sepsis')
comp_all = pd.concat([comp_neo, comp_sep])

comp_all.to_csv(os.path.join(DIR_TABLAS, 'completitud_post_limpieza.csv'), index=False)
print(comp_all.to_string(index=False))

## 4. Análisis Exploratorio

### 4.1 Distribuciones de Variables Clave

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribución de Variables por Grupo Diagnóstico', fontsize=14, fontweight='bold')

pares = [
    (df_neo, 'Neoplasia', 'dias_estadia',     'Días de Estadía',   '#2196F3', ejes[0, 0]),
    (df_sep, 'Sepsis',    'dias_estadia',     'Días de Estadía',   '#FF5722', ejes[0, 1]),
    (df_neo, 'Neoplasia', 'n_procedimientos', 'N° Procedimientos', '#2196F3', ejes[1, 0]),
    (df_sep, 'Sepsis',    'n_procedimientos', 'N° Procedimientos', '#FF5722', ejes[1, 1]),
]
for df, etq, col, xlabel, color, ax in pares:
    vals = df[col].dropna()
    ax.hist(vals, bins=50, density=True, alpha=0.6, color=color)
    vals.plot.kde(ax=ax, color='navy' if 'Neo' in etq else 'darkred', linewidth=2)
    ax.axvline(vals.median(), color='green', linestyle='--', linewidth=1.2,
               label=f'Mediana={vals.median():.1f}')
    ax.set_title(f'{etq}: {col}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '01_distribuciones.png'), dpi=DPI, bbox_inches='tight')
plt.show()
print('Ambas distribuciones son asimétricas a la derecha → justifica pruebas no paramétricas.')

### 4.2 Boxplots por Severidad GRD

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Días de Estadía por Nivel de Severidad GRD', fontsize=13, fontweight='bold')

paleta = {1: '#4CAF50', 2: '#FFC107', 3: '#FF5722', 4: '#9C27B0'}

for ax, df, etq in [(ejes[0], df_neo, 'Neoplasia'), (ejes[1], df_sep, 'Sepsis')]:
    tmp = df[[COL_SEVERIDAD, 'dias_estadia']].dropna().copy()
    tmp[COL_SEVERIDAD] = pd.to_numeric(tmp[COL_SEVERIDAD], errors='coerce')
    tmp = tmp.dropna()
    tmp[COL_SEVERIDAD] = tmp[COL_SEVERIDAD].astype(int)
    orden = sorted(tmp[COL_SEVERIDAD].unique())
    sns.boxplot(data=tmp, x=COL_SEVERIDAD, y='dias_estadia', order=orden,
                palette=paleta, ax=ax, showfliers=False)
    ax.set_title(etq)
    ax.set_xlabel('Severidad GRD (1=Menor, 4=Extrema)')
    ax.set_ylabel('Días de Estadía')

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '03_boxplot_severidad.png'), dpi=DPI, bbox_inches='tight')
plt.show()
print('Mayor severidad → más días de estadía. Este patrón debe controlarse en H3.')

### 4.3 Variabilidad Inter-Hospital (Violinplots)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Variabilidad Inter-Hospital en Días de Estadía (Top 15)',
             fontsize=13, fontweight='bold')

for ax, df, etq in [(ejes[0], df_neo, 'Neoplasia'), (ejes[1], df_sep, 'Sepsis')]:
    top15 = df[COL_HOSPITAL].value_counts().head(15).index.tolist()
    tmp = df[df[COL_HOSPITAL].isin(top15)][[COL_HOSPITAL, 'dias_estadia']].dropna()
    # Ordenar de mayor a menor mediana para lectura intuitiva
    orden = (tmp.groupby(COL_HOSPITAL)['dias_estadia']
             .median().sort_values(ascending=False).index.tolist())
    sns.violinplot(data=tmp, x=COL_HOSPITAL, y='dias_estadia', order=orden,
                   palette='tab20', ax=ax, inner='quartile', linewidth=0.8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{etq}: Estadía por Hospital')
    ax.set_xlabel('Código Hospital')
    ax.set_ylabel('Días de Estadía')

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '04_violin_hospitales.png'), dpi=DPI, bbox_inches='tight')
plt.show()
print('La variabilidad visible entre hospitales motiva la hipótesis H1.')

## 5. Pruebas de Normalidad (Shapiro-Wilk)

Antes de aplicar Kruskal-Wallis para H1, verificamos que las variables no
siguen una distribución normal. Se usa submuestra n=5000 (semilla=42) porque
el test Shapiro-Wilk solo es válido para n ≤ 5000.

In [ ]:
def prueba_shapiro(serie, etiqueta, nombre_var):
    """Aplica Shapiro-Wilk sobre submuestra reproducible."""
    rng = np.random.default_rng(SEMILLA)
    valores = serie.dropna().values
    n = min(SHAPIRO_N, len(valores))
    muestra = rng.choice(valores, size=n, replace=False)
    W, p = stats.shapiro(muestra)
    return {
        'grupo': etiqueta, 'variable': nombre_var, 'n_muestra': n,
        'W_stat': round(W, 4),
        'p_valor': p,
        'p_display': '<0.001' if p < 0.001 else f'{p:.4f}',
        'conclusion': 'No normal' if p < ALPHA else 'Normal',
    }

resultados_sw = []
for etq, df in [('Neoplasia', df_neo), ('Sepsis', df_sep)]:
    for var in ['dias_estadia', 'n_procedimientos']:
        if var in df.columns:
            resultados_sw.append(prueba_shapiro(df[var], etq, var))

sw_df = pd.DataFrame(resultados_sw)
sw_df.to_csv(os.path.join(DIR_TABLAS, 'resultados_shapiro_wilk.csv'), index=False)

print('=== RESULTADOS SHAPIRO-WILK ===')
print(sw_df.to_string(index=False))
print('\n→ Todas las variables rechazan normalidad (p < 0.05).')
print('→ Kruskal-Wallis es apropiado para H1.')

## 6. Hipótesis H1: Variabilidad Inter-Hospital (Kruskal-Wallis)

**H0:** La distribución de n_procedimientos es igual en todos los hospitales.  
**H1:** Al menos un hospital difiere significativamente.

Rechazar H0 → los hospitales NO tratan de manera uniforme a pacientes similares.

In [ ]:
from config import MIN_CASOS

def kruskal_test(df, variable, etiqueta):
    """Kruskal-Wallis agrupando por hospital."""
    grupos = [
        grp[variable].dropna().values
        for _, grp in df.groupby(COL_HOSPITAL)
        if len(grp) >= MIN_CASOS and len(grp[variable].dropna()) >= 5
    ]
    hosps_validos = [h for h, grp in df.groupby(COL_HOSPITAL) if len(grp) >= MIN_CASOS]
    if len(grupos) < 2:
        return None
    H, p = stats.kruskal(*grupos)
    return {
        'grupo_diagnostico': etiqueta, 'variable': variable,
        'H_stat': round(H, 2), 'p_valor': p,
        'p_display': '<0.001' if p < 0.001 else f'{p:.4f}',
        'n_hospitales': len(grupos),
        'sig': sig_etiqueta(p),
        'conclusion': 'RECHAZAR H0' if p < ALPHA else 'NO RECHAZAR H0',
    }

# Ejecutar para todas las variables y grupos
filas_kw = []
for etq, df in [('Neoplasia', df_neo), ('Sepsis', df_sep)]:
    for var in ['dias_estadia', 'n_procedimientos', 'n_proc_unicos']:
        if var not in df.columns:
            continue
        fila = kruskal_test(df, var, etq)
        if fila:
            filas_kw.append(fila)

kw_df = pd.DataFrame(filas_kw)
kw_df.to_csv(os.path.join(DIR_TABLAS, 'resultados_kruskal_wallis.csv'), index=False)

print('=== RESULTADOS KRUSKAL-WALLIS (H1) ===')
print(kw_df[['grupo_diagnostico', 'variable', 'H_stat', 'p_display', 'n_hospitales', 'sig', 'conclusion']].to_string(index=False))

### Interpretación H1

Si se **rechaza H0** (p < 0.05): los hospitales exhiben variabilidad significativa en
la cantidad de procedimientos realizados para pacientes del mismo grupo diagnóstico.
Esto es consistente con Cid et al. (2024), que documenta heterogeneidad estructural
en la red hospitalaria FONASA.

Para **neoplasias** (atención programada), la variabilidad puede reflejar diferencias
en protocolos quirúrgicos o capacidad oncológica instalada.
Para **sepsis** (urgencia), refleja diferencias en capacidad de UCI y toma de decisiones clínicas.

## 7. Hipótesis H2: Mortalidad (Regresión Logística)

**Modelo:**
```
logit(P(mortalidad=1)) = α + β_proc·n_procedimientos + β_sev·C(severidad) + β_edad·edad + Σγ_h·C(hospital)
```

El signo de β_proc nos indica:
- **β > 0**: más procedimientos → más riesgo (OR > 1): los procedimientos reflejan gravedad
- **β < 0**: más procedimientos → menos riesgo (OR < 1): tratamiento intensivo es protector

In [ ]:
def ajustar_logit(df, etiqueta):
    """Ajusta regresión logística con efectos fijos por hospital."""
    cols_req = ['mortalidad', 'n_procedimientos', COL_SEVERIDAD, 'edad', COL_HOSPITAL]
    df_m = df[cols_req].dropna().copy()
    df_m[COL_SEVERIDAD] = pd.to_numeric(df_m[COL_SEVERIDAD], errors='coerce')
    df_m = df_m.dropna()
    df_m[COL_SEVERIDAD] = df_m[COL_SEVERIDAD].astype(int)

    # Solo hospitales con varianza en mortalidad (ambos resultados presentes)
    tasa = df_m.groupby(COL_HOSPITAL)['mortalidad'].mean()
    hosps_ok = tasa[(tasa > 0) & (tasa < 1)].index
    df_m = df_m[df_m[COL_HOSPITAL].isin(hosps_ok)]

    if len(df_m) < 200:
        print(f'[{etiqueta}] Pocas observaciones ({len(df_m)}), omitiendo.')
        return None

    formula = f'mortalidad ~ n_procedimientos + C({COL_SEVERIDAD}) + edad + C({COL_HOSPITAL})'
    print(f'[{etiqueta}] Ajustando logit sobre {len(df_m):,} observaciones...')
    resultado = smf.logit(formula, data=df_m).fit(method='bfgs', maxiter=300, disp=False)

    params = resultado.params
    conf = resultado.conf_int()
    pvals = resultado.pvalues

    filas = []
    for var in params.index:
        if var.startswith(f'C({COL_HOSPITAL})'):
            continue  # Omitir efectos fijos individuales por hospital
        coef = params[var]
        filas.append({
            'grupo_diagnostico': etiqueta, 'variable': var,
            'coef': round(coef, 4),
            'OR': round(np.exp(coef), 4),
            'IC95_inf': round(np.exp(conf.loc[var, 0]), 4),
            'IC95_sup': round(np.exp(conf.loc[var, 1]), 4),
            'p_valor': pvals[var],
            'p_display': '<0.001' if pvals[var] < 0.001 else f'{pvals[var]:.4f}',
            'sig': sig_etiqueta(pvals[var]),
        })

    print(f'  Pseudo-R² (McFadden): {resultado.prsquared:.4f}')
    print(f'  N = {len(df_m):,}')
    return pd.DataFrame(filas), resultado

# Ajustar para ambos grupos
tablas_logit = []
for etq, df in [('Neoplasia', df_neo), ('Sepsis', df_sep)]:
    salida = ajustar_logit(df, etq)
    if salida:
        coef_df, res = salida
        tablas_logit.append(coef_df)
        with open(os.path.join(DIR_MODELOS, f'logit_{etq.lower()}_summary.txt'), 'w') as f:
            f.write(res.summary().as_text())

if tablas_logit:
    logit_combined = pd.concat(tablas_logit, ignore_index=True)
    logit_combined.to_csv(os.path.join(DIR_TABLAS, 'coeficientes_regresion_logistica.csv'), index=False)
    print('\n=== COEFICIENTES LOGÍSTICOS (variables no-hospital) ===')
    print(logit_combined[['grupo_diagnostico','variable','coef','OR','IC95_inf','IC95_sup','p_display','sig']].to_string(index=False))

### Interpretación H2

**Si β_proc > 0 (OR > 1):** Mayor intensidad de procedimientos se asocia a mayor
riesgo de muerte. Esto es consistente con la literatura de sepsis (Munir et al., 2024),
donde más procedimientos reflejan escalada de soporte vital (ventilación mecánica,
hemodiálisis), lo que indica mayor gravedad.

**Si β_proc < 0 (OR < 1):** Mayor intensidad se asocia a menor riesgo. Esto es
esperable en neoplasias (Kamaraju et al., 2022), donde más procedimientos reflejan
tratamiento oncológico multimodal coordinado.

**Severidad:** Coeficientes positivos esperados (mayor severidad → mayor mortalidad).  
**Edad:** Coeficiente positivo esperado (mayor edad → mayor riesgo).

## 8. Hipótesis H3: Días de Estadía (OLS + Efectos Fijos)

**Modelo:**
```
dias_estadia = α + β_proc·n_procedimientos + β_sev·C(severidad) + β_edad·edad + Σγ_h·C(hospital) + ε
```

También calculamos el **VIF** para detectar multicolinealidad entre predictores.

In [ ]:
def calcular_vif(df_m, etiqueta):
    """VIF para predictores numéricos (excluye dummies de hospital)."""
    cols_num = ['n_procedimientos', 'edad', COL_SEVERIDAD]
    X = df_m[cols_num].dropna().copy()
    X[COL_SEVERIDAD] = pd.to_numeric(X[COL_SEVERIDAD], errors='coerce')
    X = X.dropna()
    Xc = add_constant(X)
    filas = []
    for i, col in enumerate(Xc.columns):
        try:
            vif_val = variance_inflation_factor(Xc.values, i)
            filas.append({'grupo': etiqueta, 'variable': col,
                          'VIF': round(vif_val, 2),
                          'interpretacion': 'OK (<5)' if vif_val < 5 else 'Moderado (5-10)' if vif_val < 10 else 'Alto (>10)'})
        except:
            pass
    return pd.DataFrame(filas)

def ajustar_ols(df, etiqueta):
    """Ajusta OLS con efectos fijos por hospital."""
    cols_req = ['dias_estadia', 'n_procedimientos', COL_SEVERIDAD, 'edad', COL_HOSPITAL]
    df_m = df[cols_req].dropna().copy()
    df_m[COL_SEVERIDAD] = pd.to_numeric(df_m[COL_SEVERIDAD], errors='coerce')
    df_m = df_m.dropna()
    df_m[COL_SEVERIDAD] = df_m[COL_SEVERIDAD].astype(int)

    if len(df_m) < 100:
        return None

    formula = f'dias_estadia ~ n_procedimientos + C({COL_SEVERIDAD}) + edad + C({COL_HOSPITAL})'
    print(f'[{etiqueta}] Ajustando OLS sobre {len(df_m):,} observaciones...')
    resultado = smf.ols(formula, data=df_m).fit()

    params = resultado.params
    conf = resultado.conf_int()
    pvals = resultado.pvalues

    filas = []
    for var in params.index:
        if var.startswith(f'C({COL_HOSPITAL})'):
            continue
        coef = params[var]
        filas.append({
            'grupo_diagnostico': etiqueta, 'variable': var,
            'coef': round(coef, 4),
            'IC95_inf': round(conf.loc[var, 0], 4),
            'IC95_sup': round(conf.loc[var, 1], 4),
            'p_valor': pvals[var],
            'p_display': '<0.001' if pvals[var] < 0.001 else f'{pvals[var]:.4f}',
            'sig': sig_etiqueta(pvals[var]),
        })

    print(f'  R² ajustado: {resultado.rsquared_adj:.4f}')
    print(f'  F-estadístico: {resultado.fvalue:.2f} (p < 0.001 = {resultado.f_pvalue < 0.001})')

    vif = calcular_vif(df_m, etiqueta)
    return pd.DataFrame(filas), resultado, vif

# Ajustar para ambos grupos
tablas_ols = []
tablas_vif = []
for etq, df in [('Neoplasia', df_neo), ('Sepsis', df_sep)]:
    salida = ajustar_ols(df, etq)
    if salida:
        coef_df, res, vif_df = salida
        tablas_ols.append(coef_df)
        tablas_vif.append(vif_df)
        with open(os.path.join(DIR_MODELOS, f'ols_{etq.lower()}_summary.txt'), 'w') as f:
            f.write(res.summary().as_text())

if tablas_ols:
    ols_combined = pd.concat(tablas_ols, ignore_index=True)
    ols_combined.to_csv(os.path.join(DIR_TABLAS, 'coeficientes_regresion_ols.csv'), index=False)
    print('\n=== COEFICIENTES OLS (variables no-hospital) ===')
    print(ols_combined[['grupo_diagnostico','variable','coef','IC95_inf','IC95_sup','p_display','sig']].to_string(index=False))

if tablas_vif:
    vif_all = pd.concat(tablas_vif)
    vif_all.to_csv(os.path.join(DIR_TABLAS, 'diagnostico_vif.csv'), index=False)
    print('\n=== DIAGNÓSTICO VIF ===')
    print(vif_all.to_string(index=False))

### Interpretación H3

**β_proc** representa el aumento esperado en días de estadía por cada procedimiento
adicional, **manteniendo constante severidad, edad y hospital**.

Un β positivo es esperado (más procedimientos → estadía más larga). La magnitud
importa: β = 0.23 significa que cada procedimiento adicional agrega ~0.23 días.

**VIF:**
- < 5: Sin problema de multicolinealidad
- 5–10: Revisión recomendada
- > 10: Coeficientes inestables

Los **efectos fijos por hospital** absorben factores institucionales no medidos
(dotación, infraestructura, protocolos), garantizando que β_proc refleje
la asociación dentro de cada hospital.

## 9. Tabla Comparativa Final

In [ ]:
# Construir tabla resumen con todos los resultados clave
filas_resumen = []

# H1 — Kruskal-Wallis
if os.path.exists(os.path.join(DIR_TABLAS, 'resultados_kruskal_wallis.csv')):
    kw = pd.read_csv(os.path.join(DIR_TABLAS, 'resultados_kruskal_wallis.csv'))
    for _, r in kw.iterrows():
        filas_resumen.append({
            'hipotesis': 'H1', 'prueba': 'Kruskal-Wallis',
            'variable': r['variable'], 'grupo': r['grupo_diagnostico'],
            'estadistico': r.get('H_stat',''), 'p_valor': r.get('p_display',''),
            'sig': r.get('sig',''), 'conclusion': r.get('conclusion',''),
        })

# H2 — Logística (solo n_procedimientos)
if tablas_logit:
    for _, r in logit_combined[logit_combined['variable']=='n_procedimientos'].iterrows():
        filas_resumen.append({
            'hipotesis': 'H2', 'prueba': 'Logística',
            'variable': 'n_proc → mortalidad', 'grupo': r['grupo_diagnostico'],
            'estadistico': r['coef'], 'p_valor': r['p_display'],
            'sig': r['sig'],
            'conclusion': f"OR={r['OR']} IC95%=[{r['IC95_inf']},{r['IC95_sup']}]",
        })

# H3 — OLS (solo n_procedimientos)
if tablas_ols:
    for _, r in ols_combined[ols_combined['variable']=='n_procedimientos'].iterrows():
        filas_resumen.append({
            'hipotesis': 'H3', 'prueba': 'OLS',
            'variable': 'n_proc → dias_estadia', 'grupo': r['grupo_diagnostico'],
            'estadistico': r['coef'], 'p_valor': r['p_display'],
            'sig': r['sig'],
            'conclusion': f"β={r['coef']} IC95%=[{r['IC95_inf']},{r['IC95_sup']}]",
        })

tabla_final = pd.DataFrame(filas_resumen)
tabla_final.to_csv(os.path.join(DIR_TABLAS, 'tabla_comparativa_neoplasia_vs_sepsis.csv'), index=False)
print('=== TABLA COMPARATIVA FINAL ===')
print(tabla_final.to_string(index=False))

## 10. Discusión e Interpretación

### 10.1 H1 — Variabilidad Inter-Hospital
Resultados Kruskal-Wallis significativos confirman que los hospitales NO tratan de
manera uniforme a pacientes similares. Esto es consistente con la heterogeneidad
estructural documentada en la red FONASA (Cid et al., 2024).

### 10.2 H2 — Mortalidad
La dirección de β_proc varía según el grupo: para sepsis se espera OR > 1
(los procedimientos reflejan escalada de gravedad, consistente con Munir et al., 2024);
para neoplasias se espera OR < 1 (más procedimientos = tratamiento multimodal coordinado,
consistente con Kamaraju et al., 2022).

### 10.3 H3 — Días de Estadía
β_proc positivo esperado en ambos grupos. La magnitud interesa: un β mayor en sepsis
indicaría que cada procedimiento de soporte vital prolonga significativamente la estadía.

### 10.4 Sesgos y Limitaciones
1. **Sesgo de selección:** Los registros con hospital o severidad vacíos excluidos pueden
   ser sistemáticamente más graves o de establecimientos con peor codificación.
2. **Sesgo de información:** Omisión de diagnósticos o procedimientos en la codificación GRD.
3. **Variables omitidas:** Dotación de RRHH, financiamiento, equipamiento — absorbidos
   parcialmente por efectos fijos, pero no completamente.
4. **Multicolinealidad:** n_procedimientos y n_proc_unicos están correlacionados por
   construcción (ver tabla VIF).
5. **Generalización:** Resultados aplican solo a hospitales FONASA públicos.

### 10.5 Implicaciones
- **Gestión hospitalaria:** Hospitales con efectos fijos negativos (menos estadía)
  pueden representar buenas prácticas transferibles; los positivos, áreas a revisar.
- **Política de salud:** La variabilidad inter-hospital observada justifica protocolos
  estandarizados para sepsis (Surviving Sepsis Campaign) a nivel nacional.
- **Investigación futura:** Análisis cualitativo en hospitales "atípicos", series
  temporales 2019-2024 para detectar tendencias post-pandemia.

In [ ]:
import glob

print('=== AVANCE 2 COMPLETADO ===')
print(f'Tablas:   {os.path.abspath(DIR_TABLAS)}')
print(f'Gráficos: {os.path.abspath(DIR_GRAFICOS)}')
print(f'Modelos:  {os.path.abspath(DIR_MODELOS)}')

for label, dir_, pat in [('Tablas', DIR_TABLAS, '*.csv'),
                          ('Gráficos', DIR_GRAFICOS, '*.png'),
                          ('Modelos', DIR_MODELOS, '*')]:
    archivos = glob.glob(os.path.join(dir_, pat))
    print(f'\n{label} ({len(archivos)} archivos):')
    for f in sorted(archivos):
        print(f'  - {os.path.basename(f)}')